**Imports**

In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

**Loading cleaned dataset**

In [19]:
df = pd.read_csv("../datasets/Processed_Datasets/framingham_clean.csv")

df.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


**Defining X and y**

In [20]:
y = df["TenYearCHD"]
X = df.drop("TenYearCHD", axis=1)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (3658, 15)
Target shape: (3658,)


**Training/validation/test**

In [21]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Testing set:", X_test.shape)

Training set: (2194, 15)
Validation set: (732, 15)
Testing set: (732, 15)


**Scaling**

In [22]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

**Building the Deep Learning Model**

In [23]:
dl_model = Sequential()

dl_model.add(Dense(16, activation='relu', input_shape=(X_train.shape[1],)))
dl_model.add(Dropout(0.2))

dl_model.add(Dense(8, activation='relu'))
dl_model.add(Dropout(0.2))

dl_model.add(Dense(1, activation='sigmoid'))

D:\tf_env\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Compiling the Model**

In [24]:
dl_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

dl_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 16)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 401 (1.57 KB)

 Trainable params: 401 (1.57 KB)

 Non-trainable params: 0 (0.00 B)

**To modify imbalanced dataset**

In [25]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights))

print(class_weights)

{0: 0.5904198062432723, 1: 3.2648809523809526}


**Training the model**

In [26]:
history = dl_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4480 - loss: 0.6873 - val_accuracy: 0.5820 - val_loss: 0.6833
Epoch 2/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5793 - loss: 0.6721 - val_accuracy: 0.6270 - val_loss: 0.6492
Epoch 3/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5770 - loss: 0.6678 - val_accuracy: 0.6270 - val_loss: 0.6449
Epoch 4/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6180 - loss: 0.6518 - val_accuracy: 0.6339 - val_loss: 0.6343
Epoch 5/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6067 - loss: 0.6533 - val_accuracy: 0.6448 - val_loss: 0.6272
Epoch 6/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5934 - loss: 0.6522 - val_accuracy: 0.6352 - val_loss: 0.6280
Epoch 7/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6144 - loss: 0.6433 - val_accuracy: 0.6393 - val_loss: 0.6216
Epoch 8/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6244 - loss: 0.6421 - val_accuracy: 0.6325 - val_loss:

**Predicting calidation and test**

In [27]:
y_val_prob = dl_model.predict(X_val)
y_test_prob = dl_model.predict(X_test)

y_val_pred = (y_val_prob >= 0.3).astype(int)
y_test_pred = (y_test_prob >= 0.3).astype(int)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


**Evaluation**

In [28]:
print("Deep Learning ANN Results")

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation F1 Score:", f1_score(y_val, y_val_pred))

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test F1 Score:", f1_score(y_test, y_test_pred))

Deep Learning ANN Results
Validation Accuracy: 0.37841530054644806
Validation F1 Score: 0.29017160686427457
Test Accuracy: 0.4030054644808743
Test F1 Score: 0.334855403348554


**Confusion Matrix and classification report**

In [29]:
print("Validation Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred))

Validation Confusion Matrix:
[[184 449]
 [  6  93]]

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.29      0.45       633
           1       0.17      0.94      0.29        99

    accuracy                           0.38       732
   macro avg       0.57      0.62      0.37       732
weighted avg       0.86      0.38      0.43       732


Test Confusion Matrix:
[[185 425]
 [ 12 110]]

Test Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.30      0.46       610
           1       0.21      0.90      0.33       122

    accuracy                           0.40       732
   macro avg       0.57      0.60      0.40       732
weighted avg       0.82      0.40      0.44       732



In [30]:
val_acc = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred)

test_acc = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

dl_results_df = pd.DataFrame({
    "Model": ["Deep Learning ANN", "Deep Learning ANN"],
    "Dataset": ["Validation", "Test"],
    "Accuracy": [val_acc, test_acc],
    "F1 Score": [val_f1, test_f1]
})

print(dl_results_df)

               Model     Dataset  Accuracy  F1 Score
0  Deep Learning ANN  Validation  0.378415  0.290172
1  Deep Learning ANN        Test  0.403005  0.334855
